# ASR-2026 Spoken Numbers Challenge — Submission Notebook

This notebook:
1. Clones the project repo from GitHub.
2. Downloads the trained weights (and KenLM if present) from a GitHub release.
3. Runs inference on the Kaggle-mounted `test/` audio.
4. Writes `submission.csv` in the format Kaggle expects.

**Fill in the two URL placeholders below** after you create your repo + release.

In [ ]:
# === Configure these ===
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_RELEASE_TAG = "v1.0"
WEIGHTS_URL = f"https://github.com/<your-user>/<your-repo>/releases/download/{GITHUB_RELEASE_TAG}/best.ckpt"
LM_URL = f"https://github.com/<your-user>/<your-repo>/releases/download/{GITHUB_RELEASE_TAG}/lm.arpa"  # optional
DATA_ROOT = "/kaggle/input/asr-2026-spoken-numbers-recognition-challenge"
TEST_CSV = f"{DATA_ROOT}/test.csv"
DECODE_METHOD = "beam_lm"  # greedy | beam | beam_lm | beam_lm_rescore

In [ ]:
!pip install -q num2words jiwer soundfile librosa resampy pydub pyyaml tqdm audiomentations

In [ ]:
!git clone {GITHUB_REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo

In [ ]:
import urllib.request, os
os.makedirs('ckpts', exist_ok=True)
urllib.request.urlretrieve(WEIGHTS_URL, 'ckpts/best.ckpt')
# Optional: LM (skip if no URL / you use greedy)
try:
    urllib.request.urlretrieve(LM_URL, 'ckpts/lm.arpa')
    HAS_LM = True
except Exception as e:
    print('LM download failed, will fall back to beam without LM:', e)
    HAS_LM = False

In [ ]:
import yaml
with open('configs/conformer_ctc.yaml') as f:
    cfg = yaml.safe_load(f)
if HAS_LM:
    cfg['decode']['lm_path'] = 'ckpts/lm.arpa'
with open('configs/runtime.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

In [ ]:
# Smoke test: inference on first 10 rows only
import pandas as pd
df = pd.read_csv(TEST_CSV).head(10)
df.to_csv('test_smoke.csv', index=False)
!python -m src.infer --config configs/runtime.yaml --ckpt ckpts/best.ckpt \
    --test-csv test_smoke.csv --data-root {DATA_ROOT} \
    --out smoke_submission.csv --method {DECODE_METHOD} \
    --batch-size 4 --num-workers 0
print(pd.read_csv('smoke_submission.csv'))

In [ ]:
# Full inference run
!python -m src.infer --config configs/runtime.yaml --ckpt ckpts/best.ckpt \
    --test-csv {TEST_CSV} --data-root {DATA_ROOT} \
    --out submission.csv --method {DECODE_METHOD} \
    --batch-size 8 --num-workers 2

In [ ]:
# Validate format and preview
import pandas as pd
df = pd.read_csv('submission.csv')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
assert (df['transcription'].between(1000, 999999)).all(), 'some predictions out of range'
df.head()